In [0]:
%sql
CREATE TABLE IF NOT EXISTS hdb_resale_prices.gold.dim_town (
  town_key BIGINT GENERATED ALWAYS AS IDENTITY,
  town STRING NOT NULL
) USING DELTA;

MERGE INTO hdb_resale_prices.gold.dim_town AS target
USING (SELECT DISTINCT town FROM hdb_resale_prices.silver.resale_prices) AS source
ON target.town = source.town
WHEN NOT MATCHED THEN
  INSERT (town) VALUES (source.town)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS hdb_resale_prices.gold.dim_block (
  block_key BIGINT GENERATED ALWAYS AS IDENTITY,
  block STRING NOT NULL,
  street_name STRING NOT NULL,
  town_key BIGINT NOT NULL
) USING DELTA;

MERGE INTO hdb_resale_prices.gold.dim_block AS target
USING (
  SELECT DISTINCT s.block, s.street_name, t.town_key
  FROM hdb_resale_prices.silver.resale_prices s
  JOIN hdb_resale_prices.gold.dim_town t ON s.town = t.town
) AS source
ON target.block = source.block
  AND target.street_name = source.street_name
  AND target.town_key = source.town_key
WHEN NOT MATCHED THEN
  INSERT (block, street_name, town_key)
  VALUES (source.block, source.street_name, source.town_key)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS hdb_resale_prices.gold.dim_flat (
  flat_key BIGINT GENERATED ALWAYS AS IDENTITY,
  flat_type STRING NOT NULL,
  flat_model STRING NOT NULL,
  storey_range STRING NOT NULL
) USING DELTA;

MERGE INTO hdb_resale_prices.gold.dim_flat AS target
USING (
  SELECT DISTINCT flat_type, flat_model, storey_range
  FROM hdb_resale_prices.silver.resale_prices
) AS source
ON target.flat_type = source.flat_type
  AND target.flat_model = source.flat_model
  AND target.storey_range = source.storey_range
WHEN NOT MATCHED THEN
  INSERT (flat_type, flat_model, storey_range)
  VALUES (source.flat_type, source.flat_model, source.storey_range)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS hdb_resale_prices.gold.dim_date (
  date_key BIGINT GENERATED ALWAYS AS IDENTITY,
  month STRING NOT NULL,
  year INT NOT NULL,
  quarter INT NOT NULL
) USING DELTA;

MERGE INTO hdb_resale_prices.gold.dim_date AS target
USING (
  SELECT DISTINCT
    month,
    CAST(SUBSTRING(month, 1, 4) AS INT) AS year,
    CAST(((CAST(SUBSTRING(month, 6, 2) AS INT) - 1) / 3) + 1 AS INT) AS quarter
  FROM hdb_resale_prices.silver.resale_prices
) AS source
ON target.month = source.month
WHEN NOT MATCHED THEN
  INSERT (month, year, quarter)
  VALUES (source.month, source.year, source.quarter)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS hdb_resale_prices.gold.fact_resale_transaction (
  town_key BIGINT,
  block_key BIGINT,
  flat_key BIGINT,
  date_key BIGINT,
  resale_price DOUBLE,
  floor_area_sqm DOUBLE,
  price_per_sqm DOUBLE
) USING DELTA;


In [0]:
fact_df = spark.sql("""
  SELECT
    t.town_key,
    b.block_key,
    f.flat_key,
    d.date_key,
    s.resale_price,
    s.floor_area_sqm,
    CAST(s.resale_price AS DOUBLE) / CAST(s.floor_area_sqm AS DOUBLE) AS price_per_sqm
  FROM hdb_resale_prices.silver.resale_prices s
  JOIN hdb_resale_prices.gold.dim_town t
    ON s.town = t.town
  JOIN hdb_resale_prices.gold.dim_block b
    ON s.block = b.block AND s.street_name = b.street_name AND t.town_key = b.town_key
  JOIN hdb_resale_prices.gold.dim_flat f
    ON s.flat_type = f.flat_type AND s.flat_model = f.flat_model
    AND s.storey_range = f.storey_range
  JOIN hdb_resale_prices.gold.dim_date d
    ON s.month = d.month
""") 

spark.sql("DROP TABLE IF EXISTS hdb_resale_prices.gold.fact_resale_transaction")
fact_df.write.mode("overwrite").saveAsTable("hdb_resale_prices.gold.fact_resale_transaction")